# Ragged-edge level conditioning — model update and validation

This update keeps the BVAR in absolute monthly differences, but adds a cumulative-level block to the Durbin–Koopman forecast state. Missing PPI months remain latent, while the future level path is imposed directly in the same joint smoother draw.

The patch preserves the current `energy_bvar_model.py`, including its optional `future_exog` workflow, and creates a timestamped backup before changing it.

In [ ]:
from pathlib import Path
import importlib.util

PROJECT_ROOT = Path.cwd()
MODEL_FILE = PROJECT_ROOT / "src" / "model" / "energy_bvar_model.py"
PATCH_FILE = PROJECT_ROOT / "apply_energy_bvar_ragged_edge_fix.py"

if not MODEL_FILE.is_file():
    raise FileNotFoundError(f"Run this notebook from the bvar-energy root: {MODEL_FILE}")
if not PATCH_FILE.is_file():
    raise FileNotFoundError(
        "Place apply_energy_bvar_ragged_edge_fix.py in the bvar-energy root first."
    )

spec = importlib.util.spec_from_file_location("ragged_edge_patch", PATCH_FILE)
patch_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(patch_module)

source = MODEL_FILE.read_text(encoding="utf-8")
if patch_module.PATCH_MARKER in source:
    print("Patch already installed:", MODEL_FILE)
else:
    updated, backup = patch_module.patch_model_file(MODEL_FILE)
    print("Updated:", updated)
    print("Backup :", backup)
    print("Restart the kernel now, then rerun the main analysis notebook up to `result`.")

## After restarting the kernel

Rerun the main heat-energy or solid-fuels notebook through the estimation cell so that `result`, `levels`, and `H` exist. Then execute the cells below.

In [ ]:
import importlib
import inspect
import sys
from pathlib import Path

MODEL_DIR = Path.cwd() / "src" / "model"
if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))

import energy_bvar_model
importlib.reload(energy_bvar_model)

from energy_bvar_model import forecast_bvar_sv_outlier

print("Fonction chargée depuis :")
print(inspect.getsourcefile(forecast_bvar_sv_outlier))
print("
State-space cumulé présent :", hasattr(energy_bvar_model, "_durbin_koopman_level_draw"))

required = {"result", "levels", "H"}
missing = sorted(name for name in required if name not in globals())
if missing:
    print("
Variables à créer dans le notebook principal avant le test :", missing)

In [ ]:
import numpy as np
import pandas as pd

if not all(name in globals() for name in ("result", "levels", "H")):
    print("Test non lancé : rerun le notebook principal jusqu'à `result`, `levels` et `H`.")
else:
    ppi_observed = levels["ppi_energy"].dropna()
    if ppi_observed.empty:
        raise ValueError("Aucune observation historique de ppi_energy.")

    last_ppi_level = float(ppi_observed.iloc[-1])
    ppi_up_10 = np.full(int(H), 1.10 * last_ppi_level)
    forecast_draws = globals().get("FORECAST_DRAWS", None)

    forecast_ppi_up_10 = forecast_bvar_sv_outlier(
        result,
        H=int(H),
        level_conditions={"ppi_energy": ppi_up_10},
        n_draws=forecast_draws,
        simulate_future_outliers=True,
        seed=2026,
    )

    ppi_index = list(forecast_ppi_up_10["variables"]).index("ppi_energy")
    future_ppi = np.asarray(
        forecast_ppi_up_10["future_level_paths"], dtype=float
    )[:, :, ppi_index]
    condition_error = float(np.max(np.abs(future_ppi - ppi_up_10[None, :])))

    tail_dates = pd.DatetimeIndex(forecast_ppi_up_10["tail_dates"])
    tail_ppi = np.asarray(
        forecast_ppi_up_10["level_paths"], dtype=float
    )[:, : len(tail_dates), ppi_index]
    missing_tail = levels["ppi_energy"].reindex(tail_dates).isna().to_numpy()

    print("Dernière observation PPI :", ppi_observed.index[-1], last_ppi_level)
    print("Dates ragged-edge PPI manquantes :", list(tail_dates[missing_tail]))
    print("Condition future :", ppi_up_10[0])
    print("Erreur maximale de conditionnement :", condition_error)
    print("Nowcasts ragged-edge finis :", bool(np.isfinite(tail_ppi[:, missing_tail]).all()))

    assert condition_error < 1e-8
    assert np.isfinite(tail_ppi[:, missing_tail]).all()
    print("
Validation réussie : le pont ragged-edge et la condition de niveau sont joints.")

### Interprétation du test

- Juin, juillet et août restent des mois non conditionnés lorsque PPI n'est pas publié.
- Le smoother les tire conjointement avec la contrainte future.
- Le chemin futur de PPI respecte la condition à la précision machine.
- Aucun `ffill`, aucun saut direct depuis la dernière observation, et aucun nowcast d'août séparé ne sont utilisés.